In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# SReality


## Scrape main website

In [5]:
URL = "https://www.sreality.cz/hledani/prodej/byty/praha-2,praha-3,praha-4?velikost=3%2B1%2C3%2Bkk%2C4%2B1%2C4%2Bkk%2C5%2B1%2C5%2Bkk%2C6-a-vice&stav=dobry-stav%2Cpo-rekonstrukci%2Cvelmi-dobry-stav&cena-do=10000000&plocha-od=65&energy_efficiency_rating_search=1%7C2%7C3&pois_in_place_distance=1&pois_in_place=13"
response = requests.get(URL)

soup = BeautifulSoup(response.text, "html.parser")

In [6]:
# Find all elements with the specified class
elements = soup.find_all(
    "a",
    class_="MuiTypography-root MuiTypography-inherit MuiLink-root MuiLink-underlineAlways css-1s6ohwi",
)

# For each element, extract href and the texts of all direct children of class 'css-adf8sc', assigning to size, location, price
results = []
for el in elements:
    href = "https://www.sreality.cz" + el.get("href")

    # Skip if not a flat detail link or if it contains 'TIP'
    if not "/detail/prodej/byt" in href or "TIP" in el.getText(strip=True):
        continue

    text_elem = el.find(class_="css-adf8sc")
    if text_elem:
        texts = [
            t.get_text(strip=True).replace("\xa0", " ")
            for t in text_elem.find_all(recursive=False)
            if not ".css" in t.get_text(strip=True)
        ]
        if not texts:
            texts = [text_elem.get_text(strip=True).replace("\xa0", " ")]
    else:
        texts = []
    # Assign to size, location, price if available
    size = texts[0] if len(texts) > 0 else None
    location = texts[1] if len(texts) > 1 else None
    price = texts[2] if len(texts) > 2 else None
    results.append({"Info": size, "Lokalita": location, "Cena": price, "URL": href})

In [7]:
df = pd.DataFrame(results)

In [8]:
# Function to parse size string into 'Dispozice' and 'Užitná plocha (m²)'
def parse_size(size_str):
    size_list = (
        size_str.replace("Prodej bytu", "")
        .replace("m²", "")
        .strip()
        .split(" ", maxsplit=1)
    )
    return size_list[0], size_list[1] + " m²"


df["Dispozice"], df["Užitná plocha"] = zip(*df["Info"].map(parse_size))

In [9]:
# Calculate price per square meter and add as new column "Cena za m²"
df["Cena za m²"] = df.apply(
    lambda row: round(
        int(row["Cena"].replace("Kč", "").replace(" ", ""))
        / int(row["Užitná plocha"].replace(" m²", ""))
        if row["Cena"] and row["Užitná plocha"]
        else None
    ),
    axis=1,
)

df

,Info,Lokalita,Cena,URL,Dispozice,Užitná plocha,Cena za m²
0,Prodej bytu 4+kk 81 m²,"Vejvanovského, Praha - Chodov",8 900 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+kk...,4+kk,81 m²,109877
1,Prodej bytu 3+1 72 m²,"Bítovská, Praha - Michle",9 149 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,72 m²,127069
2,Prodej bytu 3+1 83 m²,"Klírova, Praha - Chodov",9 599 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,83 m²,115651
3,Prodej bytu 3+1 67 m²,"Přímětická, Praha - Michle",7 995 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,67 m²,119328
4,Prodej bytu 3+1 71 m²,"Brechtova, Praha - Háje",8 380 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,71 m²,118028
5,Prodej bytu 3+1 73 m²,"Benkova, Praha - Chodov",8 299 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,73 m²,113685
6,Prodej bytu 4+kk 69 m²,"Jažlovická, Praha - Chodov",8 300 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+kk...,4+kk,69 m²,120290
7,Prodej bytu 3+1 72 m²,"Bachova, Praha - Chodov",8 500 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,72 m²,118056
8,Prodej bytu 5+kk 113 m²,Praha - Chodov,5 189 000 Kč,https://www.sreality.cz/detail/prodej/byt/5+kk...,5+kk,113 m²,45920
9,Prodej bytu 4+1 79 m²,"Ledvinova, Praha - Chodov",8 500 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+1/...,4+1,79 m²,107595


## Scrape subsequent webpages

In [100]:
def get_detail_url_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup


# soup = get_detail_url(df["URL"][0])

In [11]:
def get_energy_rating(soup):
    # MuiStack-root css-a5aig2 $ Cely panel
    # css-vcjcal # rating A
    # css-1tfalix # rating B
    # css-1c72j0r # rating C
    elements = soup.find_all("div", class_="MuiStack-root css-a5aig2")

    unique_texts = set()
    for div in elements:
        for p in div.find_all("p"):
            text = p.get_text(strip=True)
            if text:
                unique_texts.add(text)

    return " - ".join(list(unique_texts))


# get_energy_rating(soup)

In [12]:
def get_general_info(soup):
    elements = soup.find_all("div", class_="css-gvoll6")
    d = {}
    for e in elements:
        k, v = e.get_text(strip=True).replace("\xa0", " ").split(":")
        try:
            d[k] = pd.to_datetime(v, dayfirst=True)
        except ValueError:
            # print(f"Skipping {k}")
            pass

    return d


# get_general_info(soup)

In [13]:
def get_detail(url):
    detail_info_d = {}
    detail_info_d["URL"] = url

    soup = get_detail_url_page(url)

    detail_info_d |= get_general_info(soup)
    detail_info_d["Energetický štítek"] = get_energy_rating(soup)

    return detail_info_d


# get_detail(df["URL"][0])

In [14]:
import tqdm

detail_info_list = []
for url in tqdm.tqdm(df["URL"].unique()):
    detail_info_list.append(get_detail(url))

100%|██████████| 21/21 [00:12<00:00,  1.67it/s]


In [15]:
df_today = pd.merge(df, pd.DataFrame(detail_info_list), on="URL").sort_values(
    by=["Energetický štítek", "Užitná plocha", "Cena za m²"],
    ascending=[True, False, True],
)

In [44]:
df_today = df_today[
    [
        "Info",
        "Lokalita",
        "Cena",
        "URL",
        "Dispozice",
        "Užitná plocha",
        "Cena za m²",
        "Vloženo",
        "Upraveno",
        "Energetický štítek",
    ]
]

In [69]:
df_prev_yes = pd.read_excel(
    "/Users/gary/Documents/Personal/flat-search/reports/flat_search_results_test.xlsx",
    usecols=["URL", "Zájem"],
    sheet_name="Hlavní",
)
df_prev_no  = pd.read_excel("/Users/gary/Documents/Personal/flat-search/reports/flat_search_results_test.xlsx",
    usecols=["URL", "Zájem"],
    sheet_name="Vyřazené",
)

df_prev = pd.concat([df_prev_yes, df_prev_no]).drop_duplicates()


In [75]:
df_final = (
    pd.merge(df_today, df_prev, on="URL", how="left")
    .fillna("")
    .sort_values(
        by=["Zájem", "Upraveno", "Energetický štítek", "Cena za m²"],
        ascending=[False, False, False, True],
    )
)

df_final 

,Info,Lokalita,Cena,URL,Dispozice,Užitná plocha,Cena za m²,Vloženo,Upraveno,Energetický štítek,Zájem
1,Prodej bytu 3+1 83 m²,"Klírova, Praha - Chodov",9 599 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,83 m²,115651,2025-05-30,2025-09-28,C - Úsporná,NE
3,Prodej bytu 4+1 79 m²,"Ledvinova, Praha - Chodov",8 500 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+1/...,4+1,79 m²,107595,2025-09-08,2025-09-21,C - Úsporná,NE
6,Prodej bytu 4+1 75 m²,"Hviezdoslavova, Praha - Háje",8 400 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+1/...,4+1,75 m²,112000,2025-09-09,2025-09-17,C - Úsporná,NE
5,Prodej bytu 3+1 76 m²,"Štichova, Praha - Háje",7 890 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,76 m²,103816,2025-04-09,2025-08-26,C - Úsporná,NE
19,Prodej bytu 3+1 67 m²,"Bítovská, Praha - Michle",9 400 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,67 m²,140299,2025-07-15,2025-09-28,Velmi úsporná - B,ANO
15,Prodej bytu 3+1 68 m²,"Vokáčova, Praha - Michle",6 990 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,68 m²,102794,2025-09-04,2025-09-24,C - Úsporná,ANO
7,Prodej bytu 3+1 74 m²,"V jezírkách, Praha - Chodov",6 990 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,74 m²,94459,2025-03-06,2025-09-04,C - Úsporná,ANO
2,Prodej bytu 4+kk 81 m²,"Vejvanovského, Praha - Chodov",8 900 000 Kč,https://www.sreality.cz/detail/prodej/byt/4+kk...,4+kk,81 m²,109877,2025-09-23,2025-09-29,C - Úsporná,
11,Prodej bytu 3+1 72 m²,"Bítovská, Praha - Michle",9 149 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,72 m²,127069,2025-08-23,2025-09-29,C - Úsporná,
16,Prodej bytu 3+1 67 m²,"Přímětická, Praha - Michle",7 995 000 Kč,https://www.sreality.cz/detail/prodej/byt/3+1/...,3+1,67 m²,119328,2025-05-21,2025-09-27,C - Úsporná,


## Excel export

In [72]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import FormulaRule
from openpyxl.formatting.rule import CellIsRule
import pandas as pd

# Split DataFrame into two: Hlavní (Zájem != "NE") and Vyřazené (Zájem == "NE")
main_df = df_final[df_final["Zájem"] != "NE"].copy()
removed_df = df_final[df_final["Zájem"] == "NE"].copy()

excel_path = (
    "/Users/gary/Documents/Personal/flat-search/reports/flat_search_results_test.xlsx"
)

# Write both DataFrames to two sheets
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    main_df.to_excel(writer, sheet_name="Hlavní", index=False)
    removed_df.to_excel(writer, sheet_name="Vyřazené", index=False)

# Now apply formatting and data validation to both sheets
wb = openpyxl.load_workbook(excel_path)

for sheet_name in ["Hlavní", "Vyřazené"]:
    ws = wb[sheet_name]

    # --- CLICKABLE URLs ---
    # Make URL column clickable (as hyperlinks)
    url_col = None
    for idx, cell in enumerate(ws[1], 1):
        if cell.value == "URL":
            url_col = idx
            break

    if url_col:
        col_letter = openpyxl.utils.get_column_letter(url_col)
        for row in range(2, ws.max_row + 1):
            cell = ws[f"{col_letter}{row}"]
            if (
                cell.value
                and isinstance(cell.value, str)
                and cell.value.startswith("http")
            ):
                cell.hyperlink = cell.value
                cell.style = "Hyperlink"

        print(f"Processed clickable URLs in column {col_letter}.")


    # --- FORMATTING AND STYLES ---
    # Define styles
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill("solid", fgColor="4F81BD")  # Blue header
    odd_fill = PatternFill("solid", fgColor="DCE6F1")  # Light blue
    even_fill = PatternFill("solid", fgColor="FFFFFF")  # White

    # Set header style
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )
    print("Styled header row.")


    # Set row styles and adjust row height
    for i, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        fill = odd_fill if i % 2 else even_fill
        for cell in row:
            cell.fill = fill
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        ws.row_dimensions[i].height = 30  # Adjust as needed for visibility
    print("Styled data rows.")


    # Auto-adjust column widths
    for col in ws.columns:
        max_length = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                cell_length = len(str(cell.value))
                if cell_length > max_length:
                    max_length = cell_length
            except:
                pass
        ws.column_dimensions[col_letter].width = max(15, min(max_length + 2, 50))
    print("Auto-adjusted column widths.")


    # --- DATA VALIDATION ---
    # Add data validation for 'Zájem' column if present
    interest_col = None
    for idx, cell in enumerate(ws[1], 1):
        if cell.value == "Zájem":
            interest_col = idx
            break

    if interest_col:
        try:
            col_letter = openpyxl.utils.get_column_letter(interest_col)
            dv = DataValidation(type="list", formula1='"ANO,NE"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f"{col_letter}2:{col_letter}{ws.max_row}")
            print(
                f"Added data validation to column {col_letter}."
            )
        except Exception as e:
            print(f"Error adding data validation: {e}")

wb.save(excel_path)
print(
    f"Excel file '{excel_path}' created with two sheets ['Hlavní', 'Vyřazené'] and formatting."
)

Excel file '/Users/gary/Documents/Personal/flat-search/reports/flat_search_results_test.xlsx' created with two sheets ('Hlavní', 'Vyřazené') and formatting.


In [ ]:

df_today['Novinka'] = df_today["URL"].apply(lambda x: "NOVINKA!" if not x in df_prev["URL"].unique() else "")
# df_today


In [87]:
import pync

# Open Finder folder when notification clicked
pync.notify(
    "Daily job finished 🚀",
    title="Python Alert",
    open="/Users/gary/Documents/Personal/flat-search/notebooks"
)

# BEZ REALITKY
## Scrape main website

In [ ]:
BEZ_REALITKY_MAIN_URL = "https://www.bezrealitky.cz/vyhledat?condition=VERY_GOOD&condition=NEW&condition=GOOD&condition=AFTER_RECONSTRUCTION&disposition=DISP_3_KK&disposition=DISP_3_1&disposition=DISP_4_KK&disposition=DISP_4_1&disposition=DISP_5_KK&disposition=DISP_5_1&disposition=DISP_6_KK&disposition=DISP_6_1&disposition=DISP_7_KK&disposition=DISP_7_1&estateType=BYT&location=exact&offerType=PRODEJ&osm_value=obvod+Praha+4%2C+Hlavn%C3%AD+m%C4%9Bsto+Praha%2C+Praha%2C+%C4%8Cesko&priceTo=10000000&regionOsmIds=R17450429&surfaceFrom=65&currency=CZK"

soup = get_detail_url_page(BEZ_REALITKY_MAIN_URL)

In [103]:
elements = soup.find_all(
    "article",
    class_="PropertyCard_propertyCard__moO_5 propertyCard PropertyCard_propertyCard--landscape__XvPmC PropertyCard_propertyCard--disable-link-mask__iJQGG",
)

In [109]:
def scrape_bezrealitky_main_page(elements):
    """
    For a list of BeautifulSoup element nodes (e.g., property cards),
    extract:
        - Info, location, href from h2 with class 'PropertyCard_propertyCardHeadline___diKI'
        - layout and size from features list with class 'FeaturesList_featuresList__75Wet featuresList mt-3'
        - price and price per metre from div with class 'PropertyPrice_propertyPrice__lthza propertyPrice mb-0 mt-3'
        - additional text from div with class 'PropertyCard_propertyCardImageHolder__Dxypp'

    Returns a list of dicts named `results`.
    """
    results = []
    for el in elements:

        # Extract info, location, href from h2
        info = None
        location = None
        href = None
        h2 = el.find(class_="PropertyCard_propertyCardHeadline___diKI")
        if h2:
            info = h2.get_text(separator=" | ", strip=True)
            location = info.split(" | ")[-1] if " | " in info else ""
            a = h2.find("a")
            if a:
                href = a.get("href")

        # Extract layout and size from features list
        features = el.find(class_="FeaturesList_featuresList__75Wet featuresList mt-3")
        if features:
            features_text = features.get_text(separator="|", strip=True)
        layout = features_text.split("|")[0] if "|" in features_text else ""
        size_m2 = (
            features_text.split("|")[-1].replace("\xa0", " ")
            if "|" in features_text
            else ""
        )

        # Extract price and price per metre from features list
        prices = el.find(
            class_="PropertyPrice_propertyPrice__lthza propertyPrice mb-0 mt-3"
        )
        if prices:
            prices_text = (
                prices.get_text(
                    separator="|",
                    strip=True,
                )
                .replace("\xa0", " ")
                .replace("|(", "")
                .replace("|)", "")
            )
        price = prices_text.split("|")[0] if "|" in prices_text else ""
        price_per_metre = int(prices_text.split("|")[-1].replace(" ", "").replace("Kč/m²", "")) if "|" in prices_text else ""

        results.append(
            {
                "Info": info,
                "Lokalita": location,
                "Cena": price,
                "URL": href,
                "Dispozice": layout,
                "Užitná plocha": size_m2,
                "Cena za m²": price_per_metre,
            }
        )

    return results

In [110]:
results = scrape_bezrealitky_main_page(elements)
df_br = pd.DataFrame(results)
df_br.head()

,Info,Lokalita,Cena,URL,Dispozice,Užitná plocha,Cena za m²
0,"Prodej bytu | Oldřichova, Praha - Nusle","Oldřichova, Praha - Nusle",9 990 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,97 m²,102990
1,"Prodej bytu | Bachova, Praha - Chodov","Bachova, Praha - Chodov",7 800 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,69 m²,113043
2,"Prodej bytu | Urbánkova, Praha - Modřany","Urbánkova, Praha - Modřany",8 999 900 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,4+kk,80 m²,112499
3,"Prodej bytu | Mařatkova, Praha - Kamýk","Mařatkova, Praha - Kamýk",9 100 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,4+1,81 m²,112346
4,"Prodej bytu | Bítovská, Praha - Michle","Bítovská, Praha - Michle",9 400 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,67 m²,140299


## Scrape subsequent webpages

In [111]:
soup = get_detail_url_page(df_br['URL'].unique()[0])

In [146]:
def get_general_info(soup):
    energy_rating = ''

    for tbody in soup.find_all("tbody"):
        trs = tbody.find_all("tr")
        for tr in trs:
            row_text = tr.get_text(strip=True, separator="|")

            # PARSE ONLY ENERGY RATING NOW
            if "PENB" in row_text:
                energy_rating = row_text.split("|")[-1]

    return {"Energetický štítek": energy_rating}


get_general_info(soup)

{'Energetický štítek': 'D - Méně úsporná'}

In [147]:
def get_detail(url):
    detail_info_d = {}
    detail_info_d["URL"] = url
    soup  = get_detail_url_page(url)
    detail_info_d |= get_general_info(soup)

    return detail_info_d

In [153]:
import tqdm
import re

detail_info_list = []
for url in tqdm.tqdm(df_br["URL"].unique(), desc="Processing detail pages"):
    details = get_detail(url)

    unapproved_energy_ratings = re.compile("D|E|F|G|H")
    if not unapproved_energy_ratings.search(details['Energetický štítek']):
        detail_info_list.append(details)

Processing detail pages: 100%|██████████| 12/12 [00:05<00:00,  2.20it/s]


In [152]:
df_br_today = pd.merge(df_br, pd.DataFrame(detail_info_list), on="URL", how="right").sort_values(
    by=["Energetický štítek", "Užitná plocha", "Cena za m²"],
    ascending=[True, False, True],
)
df_br_today

,Info,Lokalita,Cena,URL,Dispozice,Užitná plocha,Cena za m²,Energetický štítek
2,"Prodej bytu | Mařatkova, Praha - Kamýk","Mařatkova, Praha - Kamýk",9 100 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,4+1,81 m²,112346,B - Velmi úsporná
5,"Prodej bytu | Soukalova, Praha - Modřany","Soukalova, Praha - Modřany",9 990 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+kk,78 m²,128077,B - Velmi úsporná
0,"Prodej bytu | Bachova, Praha - Chodov","Bachova, Praha - Chodov",7 800 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,69 m²,113043,B - Velmi úsporná
3,"Prodej bytu | Bítovská, Praha - Michle","Bítovská, Praha - Michle",9 400 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,67 m²,140299,B - Velmi úsporná
7,"Prodej bytu | Klírova, Praha - Chodov","Klírova, Praha - Chodov",9 600 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,m²,121519,C - Úsporná
1,"Prodej bytu | Urbánkova, Praha - Modřany","Urbánkova, Praha - Modřany",8 999 900 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,4+kk,80 m²,112499,C - Úsporná
4,"Prodej bytu | Bítovská, Praha - Michle","Bítovská, Praha - Michle",9 150 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,72 m²,127083,C - Úsporná
8,"Prodej bytu | Jílovská, Praha - Braník","Jílovská, Praha - Braník",8 990 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,70 m²,128429,C - Úsporná
6,"Prodej bytu | V Jezírkách, Praha - Chodov","V Jezírkách, Praha - Chodov",7 220 000 Kč,https://www.bezrealitky.cz/nemovitosti-byty-do...,3+1,69 m²,104638,C - Úsporná
